# Tutorial: Searching Donors with PULSAR Embeddings

This tutorial demonstrates how to:
1. Load a pre-trained PULSAR model
2. Extract zero-shot donor-level embeddings from your single-cell data
3. Query a large DONORxEMBED of pre-computed donor embeddings to find similar donors
4. Use DuckDB to perform efficient similarity search on Hugging Face datasets

## Overview

PULSAR creates donor-level representations by aggregating single-cell embeddings. Once you have a donor embedding, you can search for similar donors in large databases to find matching immune profiles, discover relevant cohorts, or identify donors with similar biological characteristics. In this example, we demonstrate how to extract embeddings from a **lupus dataset** ([Perez et al.](https://www.science.org/doi/10.1126/science.abf1970)) and query the **PULSAR_DONORxEMBED** database on HuggingFace ([here](https://huggingface.co/datasets/KuanP/PULSAR_DONORxEMBED_zero_shot)), which contains pre-computed embeddings from thousands of donors across multiple studies. 

Example data used in this tutorial can be downloaded from [here](https://drive.google.com/file/d/1ln3d5Nd0hkMSMfVR_bLaRKCrU3yikEVd/view?usp=sharing).

## 1. Setup and Imports

First, let's import the necessary libraries. We'll use DuckDB for efficient vector similarity search directly on HuggingFace datasets.

In [9]:
import scanpy as sc
import numpy as np
import torch
import duckdb
from typing import List
import matplotlib.pyplot as plt
import seaborn as sns

from pulsar.utils import extract_donor_embeddings_from_h5ad
from pulsar.model import PULSAR

%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)
""

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


''

## 2. Load Single-Cell Data

We'll use a lupus dataset as our query data. This contains single-cell RNA-seq data from PBMC samples of both **lupus patients** and **healthy controls**. We'll extract embeddings from these donors and then search for similar donors in the reference database. Example data used in this tutorial can be downloaded from [here](https://drive.google.com/file/d/1ln3d5Nd0hkMSMfVR_bLaRKCrU3yikEVd/view?usp=sharing).

In [10]:
# Load the h5ad file containing single-cell data + UCE embeddings
data_path = "/lfs/blackwell1/0/kuanpang/pulsar-dev/PULSAR-release-dev/benchmark_data/lupus_subsampled_uce_adata.h5ad"
adata = sc.read_h5ad(data_path)

print(f"Dataset shape: {adata.shape}")
print(f"Number of unique donors: {adata.obs['donor_id'].nunique()}")

Dataset shape: (315919, 1902)
Number of unique donors: 261


In [11]:
# Examine the data structure
adata

AnnData object with n_obs × n_vars = 315919 × 1902
    obs: 'library_uuid', 'assay_ontology_term_id', 'mapped_reference_annotation', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'cell_state', 'sample_uuid', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_state', 'suspension_enriched_cell_types', 'suspension_uuid', 'suspension_type', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'organism_ontology_term_id', 'disease_ontology_term_id', 'sex_ontology_term_id', 'Processing_Cohort', 'ct_cov', 'ind_cov', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_cells'
    uns: 'citation', 'default_embedding', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_uce'

In [27]:

# use groupby to get the mapping of donor_id to disease
donor_id_to_disease = adata.obs.groupby('donor_id')['disease'].first().to_dict()

/tmp/user/24194/ipykernel_1249816/1845180322.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  donor_id_to_disease = adata.obs.groupby('donor_id')['disease'].first().to_dict()


## 3. Load Pre-trained PULSAR Model

Load the pre-trained PULSAR model from 🤗HuggingFace and prepare it for inference. We use the `pulsar-pbmc` model, which was trained on PBMC data and is well-suited for analyzing immune cell populations relevant to lupus.



In [12]:
# Load the model from the huggingface model hub
pulsar_model = PULSAR.from_pretrained("KuanP/pulsar-pbmc")


# Set model to evaluation mode and move to GPU
pulsar_model.eval()
pulsar_model.cuda()

# Convert to bfloat16 for efficiency
pulsar_model = pulsar_model.to(torch.bfloat16)

print("Model loaded successfully!")

Model loaded successfully!


## 4. Extract Donor Embeddings (Query Embeddings)

Use the PULSAR model to extract donor-level embeddings from your single-cell data. These embeddings will serve as **query vectors** to search the reference database. Each embedding captures the donor's immune cell composition and state in a 512-dimensional vector space.

In [13]:
# Extract donor embeddings
donor_embedding_collection = extract_donor_embeddings_from_h5ad(
    adata,
    donor_id_key="donor_id",
    model=pulsar_model,
    sample_cell_num=1024
)

print(f"Extracted embeddings for {len(donor_embedding_collection)} donors")
# Show example of embedding structure
first_donor_id = list(donor_embedding_collection.keys())[0]
print(f"\nExample donor ID: {first_donor_id}")
print(f"Embedding shape: {donor_embedding_collection[first_donor_id]['embedding'][0].shape}")

Resample 0 time


0it [00:00, ?it/s]

27it [00:00, 47.67it/s]

Extracted embeddings for 261 donors

Example donor ID: 1404
Embedding shape: (512,)


## 5. Use Cases

This similarity search capability enables several applications:

1. **Cohort Discovery**: Find donors with similar immune profiles across multiple studies
2. **Disease Matching**: Identify donors with similar disease-associated signatures
3. **Quality Control**: Detect batch effects by finding unexpected similarities
4. **Transfer Learning**: Find relevant reference donors for downstream analyses
5. **Biological Discovery**: Identify donors sharing specific immune states or responses

The PULSAR DONORxEMBED database provides a powerful resource for comparing your donors against thousands of reference samples.

In [ ]:
def similarity_search_donor_embeddings(
    query: np.ndarray,
    k: int = 5,
    dataset_name: str = "KuanP/PULSAR_DONORxEMBED_zero_shot",
    embedding_column: str = "embedding",
):
    """
    Search for similar donors in the PULSAR DONORxEMBED database.
    
    Args:
        query: Query embedding vector (512-dimensional numpy array)
        k: Number of most similar donors to return
        dataset_name: Hugging Face dataset name containing pre-computed embeddings
        embedding_column: Name of the column containing embeddings
    
    Returns:
        DataFrame with top-k similar donors and their metadata
    """
    query_vector = query
    embedding_dim = 512

    # SQL query to compute cosine distance and retrieve top-k matches
    sql = f"""
        SELECT 
            *,
            array_cosine_distance(
                {embedding_column}::float[{embedding_dim}], 
                {query_vector.tolist()}::float[{embedding_dim}]
            ) as distance
        FROM 'hf://datasets/{dataset_name}/**/*.parquet'
        ORDER BY distance
        LIMIT {k}
    """
    
    return duckdb.sql(sql).to_df()

Similarity search function defined!


## 6. Interpret Search Results

The results show donors with similar immune profiles to your query donor. Key columns include:
- **distance**: Lower values indicate higher similarity (cosine distance)
- **donor_id**: Unique identifier for each donor in the database
- **study**: Source dataset/study of the donor
- **disease** or other metadata: Phenotypic information about the donor

Let's examine the distance distribution to understand how similar these donors are.

In [28]:
# Get the embedding for the first donor in our dataset
query_donor_id = first_donor_id
query_embedding = donor_embedding_collection[query_donor_id]['embedding'][0]
disease_label = donor_id_to_disease[query_donor_id]

print(f"Querying database with donor: {query_donor_id} with disease {disease_label}")
print(f"Query embedding shape: {query_embedding.shape}")

# Search for top 5 similar donors
results = similarity_search_donor_embeddings(
    query=query_embedding,
    k=5
)

print(f"\nFound {len(results)} similar donors:")
results

Querying database with donor: 1404 with disease systemic lupus erythematosus
Query embedding shape: (512,)

Found 5 similar donors:


,donor_id,embedding,disease_label,dataset_id,distance
0,1404_218acb0f-9f2f-4f76-b90b-15a4b7c7f629,"[-0.0024132216, 0.6152637, 0.06741905, 0.46352...",systemic lupus erythematosus,218acb0f-9f2f-4f76-b90b-15a4b7c7f629,0.000879
1,1418_218acb0f-9f2f-4f76-b90b-15a4b7c7f629,"[0.06743599, 0.7286544, -0.053424273, 0.415710...",systemic lupus erythematosus,218acb0f-9f2f-4f76-b90b-15a4b7c7f629,0.007418
2,1602_218acb0f-9f2f-4f76-b90b-15a4b7c7f629,"[0.10306098, 0.45203608, 0.02977808, 0.4375849...",systemic lupus erythematosus,218acb0f-9f2f-4f76-b90b-15a4b7c7f629,0.011851
3,1366_218acb0f-9f2f-4f76-b90b-15a4b7c7f629,"[0.15112329, 0.6606278, 0.060532592, 0.4318714...",systemic lupus erythematosus,218acb0f-9f2f-4f76-b90b-15a4b7c7f629,0.012859
4,1222_218acb0f-9f2f-4f76-b90b-15a4b7c7f629,"[-0.10630939, 0.676438, 0.011803958, 0.2359959...",systemic lupus erythematosus,218acb0f-9f2f-4f76-b90b-15a4b7c7f629,0.013227


Looks like our query donor is most similar to other lupus patients, which aligns with our expectations given the disease context!

## 7. Next Steps

We provided different models (including `PULSAR-aligned` and `PULSAR_DONORxEMBED_aligned`) that specialize in disease mapping and cross-study alignment. You can explore these models for more advanced analyses.